# 02 - Feature engineering

Fluxes to AB magnitudes (zeropoint 25), all pairwise colors. Missing bands (flux <= 0, a small fraction per band) are median-imputed rather than dropped -- dropping would lose ~40% of rows to one band (F814W) alone.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

PROC_DIR = Path("../data/processed")
PROC_DIR.mkdir(parents=True, exist_ok=True)
BANDS = ["F606W", "F814W", "F125W", "F140W", "F160W"]

df = pd.read_parquet("../data/raw/3dhst_master_clean.parquet")

In [2]:
# flux -> AB mag; invalid flux (<=0, the catalog's missing-data sentinel) -> NaN
for b in BANDS:
    flux = df[f"f_{b}"].where(df[f"f_{b}"] > 0)
    df[f"mag_{b}"] = 25.0 - 2.5 * np.log10(flux)

# median-impute missing magnitudes (simple constant fill, not a fitted pipeline)
mag_cols = [f"mag_{b}" for b in BANDS]
df[mag_cols] = df[mag_cols].fillna(df[mag_cols].median())
df[mag_cols].isna().sum()

mag_F606W    0
mag_F814W    0
mag_F125W    0
mag_F140W    0
mag_F160W    0
dtype: int64

In [3]:
import itertools

# all pairwise colors
for b1, b2 in itertools.combinations(BANDS, 2):
    df[f"color_{b1}_{b2}"] = df[f"mag_{b1}"] - df[f"mag_{b2}"]

FEATURE_COLS = mag_cols + [f"color_{b1}_{b2}" for b1, b2 in itertools.combinations(BANDS, 2)]
len(FEATURE_COLS), FEATURE_COLS

(15,
 ['mag_F606W',
  'mag_F814W',
  'mag_F125W',
  'mag_F140W',
  'mag_F160W',
  'color_F606W_F814W',
  'color_F606W_F125W',
  'color_F606W_F140W',
  'color_F606W_F160W',
  'color_F814W_F125W',
  'color_F814W_F140W',
  'color_F814W_F160W',
  'color_F125W_F140W',
  'color_F125W_F160W',
  'color_F140W_F160W'])

In [4]:
keep = ["id", "field", "ra", "dec", "z_spec", "z_peak"] + FEATURE_COLS
df[keep].to_parquet(PROC_DIR / "features.parquet", index=False)
import json
json.dump(FEATURE_COLS, open(PROC_DIR / "feature_cols.json", "w"))
df[keep].shape

(3971, 21)